In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
class_mapping = {
    'hc': 0,
    'CIS': 1,
    'RRMS': 2,
    'SPMS': 3,
    'PPMS': 4,
}

root = Path('/media/yannik/Intenso/DATA/dfg_plexus')
# root = Path('/run/media/yannik/Intenso/DATA/dfg_plexus')
# root = Path('F:/DATA/dfg_plexus/')

hc_df = pd.read_csv(root / "radiomics_features_hc.csv", delimiter=";").rename(columns={'Unnamed: 0': 'patID'})
ms_df = pd.read_csv(root / "radiomics_features_ms.csv", delimiter=";").rename(columns={'Unnamed: 0': 'patID'})
ms_meta = pd.read_excel(root / "MSData_noNMO.xlsx")
ms_df = ms_df.merge(ms_meta[['patID', 'Disease Course/ Type at MRI date']],
                    on='patID', how='left')

ms_df['label'] = ms_df['Disease Course/ Type at MRI date'].map(class_mapping)
hc_df['label'] = 0

df = pd.concat([hc_df, ms_df], ignore_index=True)
df = df.drop(columns=['Disease Course/ Type at MRI date'])

# Check for missing values after mapping
unmatched = ms_df[ms_df['label'].isna()]

# Drop nan rows
df = df.dropna(subset=['label'])

df.to_csv(root / "radiomics_features___combined.csv", sep=";", index=False)

ft = df.drop(columns=['label', 'patID'])  # Shape (N_patients, N_features)
label = df['label'].astype(int)  # Shape (N_patients, )

In [ ]:
ft.shape

In [ ]:
df.shape

In [ ]:
_df = pd.read_csv(root / "radiomics_features___combined.csv", sep=";")
_df.shape

# Label Distribution

In [ ]:
label.value_counts().sort_index().plot(kind='bar')
plt.xlabel('Class label')
#plt.xticks(ticks=class_mapping.values() ,labels=class_mapping.keys())
plt.ylabel('Number of subjects')
plt.title('Label distribution')
plt.show()

print(label.value_counts(normalize=True))

# Optional: Feature Normalisation

In [ ]:
from sklearn.preprocessing import StandardScaler
# ft = pd.DataFrame(StandardScaler().fit_transform(ft), columns=ft.columns, index=ft.index)

# Checking for NANs etc.

In [ ]:
missing_counts = ft.isna().sum()
print("Columns with missing values:\n", missing_counts[missing_counts > 0])

print("Non-numeric columns:", ft.select_dtypes(exclude=[np.number]).columns.tolist())

# Feature Scaling and Variance

In [ ]:
low_variance = ft.var()[ft.var() < 0.005]
print(f"Number of low variance features: {len(low_variance)}")

In [ ]:
high_variance = ft.var()[ft.var() > 100.00]
print(f"Number of high variance features: {len(high_variance)}")

In [ ]:
# Remove low variance features 
# ft = ft.drop(columns=low_variance.index, errors="ignore")
# ft.shape

In [ ]:
# ft.to_csv(root / "radiomics_features___combined___no_low_var.csv", sep=";")

In [ ]:
ft.var().sort_values(ascending=False).head(10)

# Feature Redundancy or Correlation

In [ ]:
corr = ft.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr_features = [column for column in upper.columns if any(upper[column] > 0.95)]
print(f"Highly correlated features (>0.95): {len(high_corr_features)}")

In [ ]:
# Clustering-based feature grouping

from scipy.cluster.hierarchy import linkage, fcluster
import numpy as np

# Compute correlation distance
corr = ft.corr().abs()
distance = 1 - corr

# Hierarchical clustering
link = linkage(distance, method='average')
cluster_ids = fcluster(link, 0.95, criterion='distance')

# Pick one representative feature per cluster
representatives = [ft.columns[np.where(cluster_ids == i)[0][0]] for i in np.unique(cluster_ids)]
ft = ft[representatives]
ft.shape

In [ ]:
# ft.to_csv(root / "radiomics_features___combined___no_low_var__no_corr.csv", sep=";")
# ft.to_csv(root / "radiomics_features___combined___no_corr.csv", sep=";")

# Separability Check

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
components = pca.fit_transform(ft)
plt.scatter(components[:, 0], components[:, 1], c=label, cmap='tab10', alpha=0.7)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA projection of radiomics features')
plt.show()

# Separability MS vs HC

In [ ]:
binary_label = (label != 0).astype(int)  # 0 = HC, 1 = MS
print(binary_label.value_counts())

In [ ]:
y_label = binary_label

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2, random_state=42)
components = pca.fit_transform(ft)

plt.figure(figsize=(7,5))
plt.scatter(components[:, 0], components[:, 1], c=y_label, cmap='coolwarm', alpha=0.7)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA projection of Choroid Plexus radiomics (HC=0, MS=1)')
plt.colorbar(label='MS label')
plt.show()

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42)
embedding = tsne.fit_transform(ft)

plt.figure(figsize=(7,5))
plt.scatter(embedding[:,0], embedding[:,1], c=y_label, cmap='coolwarm', alpha=0.7)
plt.title('t-SNE embedding (HC vs MS)')
plt.show()

# Label -> Feature Leakage Check (Binary; HC vs. MS)

In [ ]:
assert ft.index.equals(y_label.index)
assert np.isfinite(y_label.to_numpy()).all()

In [ ]:
from scipy.stats import ttest_ind

ttest_results = []

mask_hc = (binary_label == 0)  # aligned Series
mask_ms = ~mask_hc

for f in ft.columns:
    s = ft[f]
    # Use 1D boolean indexing on the Series itself (avoids the .loc tuple path)
    hc_values = s[mask_hc]
    ms_values = s[mask_ms]
    stat, p = ttest_ind(hc_values, ms_values, equal_var=False, nan_policy='omit')
    ttest_results.append((f, p))

significant = [f for f, p in ttest_results if p < 1e-5]
print(f"Features showing strong univariate separation (p < 1e-5): {len(significant)}")
print(significant[:10])

# Quick Baseline Classifier

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(ft, label, stratify=label,
                                                    test_size=0.3, random_state=42)

n_classes = np.unique(y_test.values).size
print(f"{n_classes=}")

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    objective='binary:logistic' if n_classes == 2 else 'multi:softprob',
    num_class=None if n_classes == 2 else n_classes,
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("var", VarianceThreshold(0.001)),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(penalty='l2', solver='lbfgs', max_iter=10000))
])

model.fit(X_train, y_train)

In [ ]:
# Testset Evaluation
from scipy.special import softmax
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import cross_val_score

if n_classes == 2:
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_proba))
else:
    try:
        logits = model.predict(X_test, output_margin=True)
        proba = softmax(logits, axis=1)
        y_pred = proba.argmax(axis=1)
    except TypeError:
        y_pred = model.predict(X_test)

    print(classification_report(y_test, y_pred, digits=3))

    print("Confusion matrix (rows=true, cols=pred):\n", confusion_matrix(y_test, y_pred))

    # One-vs-rest ROC-AUC (macro)
    y_test_ovr = np.zeros((len(y_test), 3), dtype=int)
    for i, c in enumerate(y_test.to_numpy()):
        y_test_ovr[i, c] = 1
    auc_macro_ovr = roc_auc_score(y_test_ovr, proba, average='macro', multi_class='ovr')
    print("Macro ROC-AUC (OvR):", auc_macro_ovr)


In [ ]:
# Trainingset evaluation

if n_classes == 2:
    y_pred = model.predict(X_train)
    y_proba = model.predict_proba(X_train)[:, 1]

    print(classification_report(y_train, y_pred))
    print("ROC-AUC:", roc_auc_score(y_train, y_proba))
else:
    logits = model.predict(X_train, output_margin=True)
    proba = softmax(logits, axis=1)
    y_pred = proba.argmax(axis=1)
    print(classification_report(y_train, y_pred, digits=3))  # macro avg is what to watch

    print("Confusion matrix (rows=true, cols=pred):\n", confusion_matrix(y_train, y_pred))

    # One-vs-rest ROC-AUC (macro)
    y_train_ovr = np.zeros((len(y_train), 3), dtype=int)
    for i, c in enumerate(y_test.to_numpy()):
        y_train[i, c] = 1
    auc_macro_ovr = roc_auc_score(y_train_ovr, proba, average='macro', multi_class='ovr')
    print("Macro ROC-AUC (OvR):", auc_macro_ovr)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, ft_reduced, binary_label, cv=skf, scoring='roc_auc')
print("CV ROC-AUCs:", scores, "mean =", scores.mean(), "±", scores.std())

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, ft_reduced, binary_label, cv=skf, scoring='f1_macro')
print("CV Macro F1s:", scores, "mean =", scores.mean(), "±", scores.std())

# Feature Importance

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Get importances
importance = model.feature_importances_

# Wrap in a dataframe
importance_df = pd.DataFrame({
    'feature': ft_reduced.columns,
    'importance': importance,
    'variance': ft_reduced.var().values
}).sort_values('importance', ascending=False)

# Plot top 20
plt.figure(figsize=(8,6))
plt.barh(importance_df['feature'].iloc[:20][::-1],
         importance_df['importance'].iloc[:20][::-1])
plt.xlabel('XGBoost Feature Importance')
plt.title('Top 20 Most Important Radiomic Features (HC vs MS)')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

model = Pipeline([
    # ("var", VarianceThreshold(0.01)),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(penalty='l1', solver='saga', max_iter=1000))
])

model.fit(X_train, y_train)

In [ ]:
# Testset Evaluation
from scipy.special import softmax
from sklearn.metrics import confusion_matrix

if n_classes == 2:
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_proba))
else:
    logits = model.predict(X_test, output_margin=True)
    proba = softmax(logits, axis=1)
    y_pred = proba.argmax(axis=1)
    print(classification_report(y_test, y_pred, digits=3))

    print("Confusion matrix (rows=true, cols=pred):\n", confusion_matrix(y_test, y_pred))

    # One-vs-rest ROC-AUC (macro)
    y_test_ovr = np.zeros((len(y_test), 3), dtype=int)
    for i, c in enumerate(y_test.to_numpy()):
        y_test_ovr[i, c] = 1
    auc_macro_ovr = roc_auc_score(y_test_ovr, proba, average='macro', multi_class='ovr')
    print("Macro ROC-AUC (OvR):", auc_macro_ovr)


In [ ]:
# Trainingset evaluation

if n_classes == 2:
    y_pred = model.predict(X_train)
    y_proba = model.predict_proba(X_train)[:, 1]

    print(classification_report(y_train, y_pred))
    print("ROC-AUC:", roc_auc_score(y_train, y_proba))
else:
    logits = model.predict(X_train, output_margin=True)
    proba = softmax(logits, axis=1)
    y_pred = proba.argmax(axis=1)
    print(classification_report(y_train, y_pred, digits=3))  # macro avg is what to watch

    print("Confusion matrix (rows=true, cols=pred):\n", confusion_matrix(y_train, y_pred))

    # One-vs-rest ROC-AUC (macro)
    y_train_ovr = np.zeros((len(y_train), 3), dtype=int)
    for i, c in enumerate(y_test.to_numpy()):
        y_train[i, c] = 1
    auc_macro_ovr = roc_auc_score(y_train_ovr, proba, average='macro', multi_class='ovr')
    print("Macro ROC-AUC (OvR):", auc_macro_ovr)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, ft_reduced, binary_label, cv=skf, scoring='roc_auc')
print("CV ROC-AUCs:", scores, "mean =", scores.mean(), "±", scores.std())

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, ft_reduced, binary_label, cv=skf, scoring='f1_macro')
print("CV Macro F1s:", scores, "mean =", scores.mean(), "±", scores.std())

# Feature Importance

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Get importances
importance = model.feature_importances_

# Wrap in a dataframe
importance_df = pd.DataFrame({
    'feature': ft_reduced.columns,
    'importance': importance,
    'variance': ft_reduced.var().values
}).sort_values('importance', ascending=False)

# Plot top 20
plt.figure(figsize=(8,6))
plt.barh(importance_df['feature'].iloc[:20][::-1],
         importance_df['importance'].iloc[:20][::-1])
plt.xlabel('XGBoost Feature Importance')
plt.title('Top 20 Most Important Radiomic Features (HC vs MS)')
plt.tight_layout()
plt.show()

# Calibration Check

In [ ]:
from scipy.special import expit

raw = model.predict(X_test, output_margin=True)
probs = expit(raw)

plt.figure(figsize=(6,4))
plt.hist(probs[y_test == 0], bins=20, alpha=0.7, label='HC')
plt.hist(probs[y_test == 1], bins=20, alpha=0.7, label='MS')
plt.legend()
plt.xlabel('Predicted MS probability')
plt.ylabel('Count')
plt.title('Prediction probability distribution')
plt.show()

# Model Importance vs. Variance

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(importance_df['variance'], importance_df['importance'], alpha=0.6)
plt.xscale('log')
plt.xlabel('Feature variance (log scale)')
plt.ylabel('XGBoost importance')
plt.title('Feature Importance vs Variance (HC vs MS)')
plt.grid(True, alpha=0.3)
plt.show()